# 🎯 [WEL03_M05] 센서 오류에 의한 단수 오경보 방지 및 수질 안전 신뢰도 지수 산출 모듈 개발 (학생 실습용)
### Google Colab 실습 환경 (장병 복지·지원 / Level 2 - 학생 실습용)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/WEL03/wel03_m05_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장병 복지·지원 (Level 2) |
| **미션 ID** | WEL03_M05 |
| **미션 주제** | 센서 오류에 의한 단수 오경보 방지 및 수질 안전 신뢰도 지수 산출 모듈 개발 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
전장에서는 급수차, 정수 장비, 우물, 하천 취수원 등 급수원의 수질 이상을 빨리 알아채지 못하면 장병의 건강과 부대 활동에 바로 영향을 주지만, 센서 값은 잡음과 고장이 많아 오경보로 불필요하게 급수를 끊는 일도 생긴다. 러시아-우크라이나 전쟁 사례에서 학습 초점으로 삼은 전장 위생·건강 위험 관리를 반영해, 본 프로젝트는 가상 수질검사 데이터를 분석해 위험 신호를 데이터로 판단하는 과제를 다룬다. 교육용 가상 급수원 8곳의 pH, 탁도, 잔류염소, 중금속, 대장균 측정 자료를 코드로 만들어 스키마 정의, 센서 결측·이상치 정제, 기준치 Baseline 판정, 복합 오염 징후 분류 모델, 신뢰도 지수와 오경보 감축, 상황판 대시보드, 급수 중단 HITL 결재선, 대량 오염 부하 시험, AAR 임계치 재조정, 통합 패키지까지 단계별로 구현한다. 최종 목표는 수질 위험 신호를 근거와 함께 빠르게 제시하되, 실제 음용 판정과 급수 중단은 공식 기준과 의무(예방의학) 담당 확인, 지휘관 결심에 따르도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/wel03_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 결재 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/wel03_data_spec.md`, `docs/wel03_aar_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/wel03_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/wel03_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/wel03_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 센서 오류에 의한 단수 오경보 방지 및 수질 안전 신뢰도 지수 산출 모듈 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

센서 오류에 의한 단수 오경보 방지 신뢰도 지수 산출 모듈을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **센서 오류로 인한 급수 중단 오경보를 줄이는 수질 안전 신뢰도 지수와 High·Med·Low 등급 모듈을 개발한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **위험 경보마다 항목별 편차의 일치도, 경보 지속 시간, 센서 이상 표시, 드리프트 의심 여부를 점수화해 0~100점 신뢰도를 만들고 High·Med·Low로 나눈다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/verification/wel03_confidence_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 신뢰도 등급(High/Med/Low) 부여 기준의 군사적 타당성

#### 미션 PASS 조건

- 정량 기준: 오경보율(FAR) 40% 이상 감축, 신뢰도 스코어 산출 일관성 90% 이상


- 진행 기준: 오경보 필터링 전후 비교 지표 산출 및 통합 테스트 완료


- 검증 방법: ROC-AUC 곡선 분석 및 임계값 민감도 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""위험 경보마다 다항목 동시 변화·연속 지속·센서 상태·단일 항목 드리프트 여부로 수질 안전 신뢰도 지수(0~100, High/Med/Low)를 산출해, 센서 오류에 의한 급수 중단(단수) 오경보를 줄이고 ROC-AUC·임계값 민감도·산출 일관성을 확인한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 급수원 가상 수질 센서 데이터 생성기 (실제 부대·위치·검사 결과 없음)
# 가상 급수원 8곳 x 14일 x 1시간 간격. 실제 수질(true_*)·센서 측정값·오염 사건·센서 고장 정답을 함께 만든다.
# 기준치는 교육용 가정값이며, 실제 음용 판정은 공인 검사 결과와 의무(예방의학) 담당 기준에 따른다.
# ---------------------------------------------------------------------------
PARAMS = ["ph", "turbidity", "chlorine", "metal", "ecoli"]
SOURCES = [("W01", "water_truck", 1.0, 2.0), ("W02", "water_truck", 3.5, 1.2), ("W03", "purification_unit", 2.2, 4.1),
           ("W04", "purification_unit", 5.0, 3.3), ("W05", "well", 0.8, 5.6), ("W06", "well", 4.4, 6.0),
           ("W07", "river_intake", 6.3, 1.8), ("W08", "river_intake", 6.9, 4.9)]
BASE = {"water_truck": (7.2, 0.25, 0.75, 1.5), "purification_unit": (7.0, 0.20, 0.85, 1.2),
        "well": (6.8, 0.45, 0.50, 3.0), "river_intake": (7.3, 0.55, 0.60, 2.0)}   # pH, 탁도(NTU), 잔류염소(mg/L), 중금속(ug/L)
LIMITS = {"ph": (5.8, 8.5), "turbidity": (1.0, 2.0), "chlorine": (0.2, 0.1), "metal": (5.0, 10.0), "ecoli": (1, 1)}  # (주의, 위험) 가정 기준
EVENT_TYPES = {  # 오염 사건: 변수별 최대 변화량, 최대치 도달 시간(h)
    "sewage": ({"turbidity": 2.8, "chlorine": -0.5, "ecoli": 30, "ph": -0.3}, 4),
    "metal": ({"metal": 22.0, "ph": -0.5}, 4),
    "disinfect_fail": ({"chlorine": -0.8, "ecoli": 6}, 6),
    "chemical": ({"ph": 1.7, "turbidity": 1.0}, 3),
    "creeping_metal": ({"metal": 13.0}, 18),
}
FAULT_TYPES = ["stuck", "spike", "drift", "dropout"]
NOISE = {"ph": 0.04, "turbidity": 0.06, "chlorine": 0.03, "metal": 0.35}
DIGITS = {"ph": 2, "turbidity": 2, "chlorine": 2, "metal": 1}


def make_dataset(days=14, seed=7, n_events=6, n_faults=10, events=None, faults=None, sources=None):
    """events: [(급수원 번호, 사건 유형, 시작 시각 h, 지속 h, 강도)], faults: [(급수원 번호, 고장 유형, 항목, 시작 h, 지속 h)]"""
    rng = np.random.default_rng(seed)
    sources = sources or SOURCES
    T = days * 24
    tt = np.arange(T)
    hour = tt % 24
    night = (hour < 6) | (hour >= 20)
    rain = np.zeros(T)
    for _ in range(max(1, days // 4)):
        s0, d = int(rng.integers(0, T - 14)), int(rng.integers(6, 14))
        rain[s0:s0 + d] = np.maximum(rain[s0:s0 + d], rng.uniform(0.6, 1.0))
    if events is None:
        events = [(int(rng.integers(len(sources))), str(rng.choice(list(EVENT_TYPES))), int(rng.integers(min(24, T // 4), max(T // 4 + 1, T - 48))),
                   int(rng.integers(10, 30)), 1.0) for _ in range(n_events)]
    if faults is None:
        faults = [(int(rng.integers(len(sources))), str(rng.choice(FAULT_TYPES)), str(rng.choice(["ph", "turbidity", "chlorine", "metal"])),
                   int(rng.integers(12, max(13, T - 30))), int(rng.integers(6, 24))) for _ in range(n_faults)]
    out = []
    for si, (sid, stype, x, y) in enumerate(sources):
        ph0, tb0, cl0, mt0 = BASE[stype]
        tr = {"ph": ph0 + 0.08 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 0.03, T),
              "turbidity": tb0 * (1 + 0.1 * rng.normal(0, 1, T)),
              "chlorine": (cl0 * np.exp(-0.022 * hour) if stype == "water_truck" else cl0 + rng.normal(0, 0.03, T)),
              "metal": mt0 + rng.normal(0, 0.2, T), "ecoli": np.zeros(T)}
        if stype in ("river_intake", "well"):
            tr["turbidity"] = tr["turbidity"] + rain * (0.9 if stype == "river_intake" else 0.45)
        f_all, etype = np.zeros(T), np.array(["none"] * T, dtype=object)
        for (s_i, et, onset, dur, scale) in events:
            if s_i != si:
                continue
            delta, ramp = EVENT_TYPES[et]
            f = np.clip((tt - onset + 1) / ramp, 0, 1) * ((tt >= onset) & (tt < onset + dur + ramp))
            for k, v in delta.items():
                tr[k] = tr[k] + v * f * scale
            etype[f > 0] = et
            f_all = np.maximum(f_all, f)
        tr["chlorine"] = np.clip(tr["chlorine"], 0.02, None)
        tr["turbidity"] = np.clip(tr["turbidity"], 0.05, None)
        tr["metal"] = np.clip(tr["metal"], 0.1, None)
        grade = np.where(f_all >= 0.2, 2, 0)
        mild = (tr["turbidity"] > LIMITS["turbidity"][0]) | (tr["chlorine"] < LIMITS["chlorine"][0]) | (tr["metal"] > LIMITS["metal"][0])
        grade = np.where((grade == 0) & mild, 1, grade)
        ms = {}
        for p in ["ph", "turbidity", "chlorine", "metal"]:
            sd = NOISE[p] * np.where(night, 1.5, 1.0) * (tr[p] if p == "turbidity" else 1)
            ms[p] = np.round(np.clip(tr[p] + rng.normal(0, 1, T) * sd, 0, None), DIGITS[p])
        ms["ecoli"] = (rng.poisson(tr["ecoli"]) + (rng.random(T) < 0.004)).astype(float)
        fault, fparam = np.array(["none"] * T, dtype=object), np.array(["-"] * T, dtype=object)
        for (s_i, ft, p, s0, d) in faults:
            if s_i != si:
                continue
            seg = slice(s0, min(T, s0 + d))
            if ft == "stuck":
                ms[p][seg] = ms[p][s0]
            elif ft == "spike":
                seg = slice(s0, s0 + 1)
                ms[p][s0] = {"ph": 11.8, "turbidity": 38.0, "chlorine": 0.0, "metal": 75.0}[p]
            elif ft == "drift":
                ramp = np.linspace(0, 1, seg.stop - s0)
                ms[p][seg] = np.clip(ms[p][seg] + ramp * {"ph": -1.6, "turbidity": 2.6, "chlorine": -0.75, "metal": 12.0}[p], 0, None).round(DIGITS[p])
            elif ft == "dropout":
                for q in PARAMS:
                    ms[q][seg] = np.nan
                p = "all"
            fault[seg], fparam[seg] = ft, p
        df = pd.DataFrame({"source_id": sid, "source_type": stype, "x_km": x, "y_km": y, "t": tt,
                           "cond": np.where(rain > 0, "storm", np.where(night, "night", "day")),
                           **{p: ms[p] for p in PARAMS}, **{f"true_{p}": tr[p] for p in PARAMS},
                           "event": etype, "fault": fault, "fault_param": fparam, "grade": grade})
        out.append(df)
    return pd.concat(out, ignore_index=True)
# ---------------------------------------------------------------------------

# ---- 전처리 공통 함수: 물리 범위 점검 -> 고정값(stuck) 표시 -> 단발 스파이크 제거 -> 결측 보정 ----
PHYS = {"ph": (0, 14), "turbidity": (0, 1000), "chlorine": (0, 10), "metal": (0, 1000), "ecoli": (0, 100000)}
SPIKE_K = {"day": 6.0, "night": 7.0, "storm": 9.0}          # 상황별(주간/야간/악천후) 스파이크 판정 배수
MIN_JUMP = {"ph": 0.8, "turbidity": 1.5, "chlorine": 0.3, "metal": 6.0}
STUCK_RUN, SHORT_GAP = 6, 3


def clean(df, spike_k=None, min_jump=None):
    """급수원별 시계열을 정제하고 항목별 처리 내역(<항목>_flag: range/stuck/spike/interp/long_gap)을 남긴다."""
    # [TODO : 학생 실습 5] 센서 오류에 의한 단수 오경보 방지 및 수질 안전 신뢰도 지수 산출 모듈 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
GRADE_NAME = {0: "안전", 1: "주의", 2: "위험"}


def rule_grade(df):
    """기준치 1차 필터: 위험(음용 부적합 의심) = 한 항목이라도 위험 기준 초과, 주의 = 주의 기준 초과."""
    # [TODO : 학생 실습 5] 센서 오류에 의한 단수 오경보 방지 및 수질 안전 신뢰도 지수 산출 모듈 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def reasons(row):
    r = []
    if not LIMITS["ph"][0] <= row["ph"] <= LIMITS["ph"][1]:
        r.append(f"pH {row['ph']:.2f}")
    if row["turbidity"] > LIMITS["turbidity"][0]:
        r.append(f"탁도 {row['turbidity']:.2f}NTU")
    if row["chlorine"] < LIMITS["chlorine"][0]:
        r.append(f"잔류염소 {row['chlorine']:.2f}mg/L")
    if row["metal"] > LIMITS["metal"][0]:
        r.append(f"중금속 {row['metal']:.1f}ug/L")
    if row["ecoli"] >= 1:
        r.append(f"대장균 {row['ecoli']:.0f}")
    return ", ".join(r) or "-"


FEATS = [f"{p}{s}" for p in ["ph", "turbidity", "chlorine", "metal"] for s in ["", "_d1", "_d3", "_dev", "_sd6"]] + \
        ["ecoli", "ecoli_3h", "storm", "night", "n_flag", "truck", "purif", "well", "river"]


def add_features(c):
    """시간 순서를 지키는(과거 값만 쓰는) 급수원별 특징: 변화량, 24시간 기준선 대비 편차, 6시간 변동성, 대장균 누적 등."""
    c = c.sort_values(["source_id", "t"]).reset_index(drop=True)
    g = c.groupby("source_id")
    for p in ["ph", "turbidity", "chlorine", "metal"]:
        c[f"{p}_d1"] = g[p].diff().fillna(0)
        c[f"{p}_d3"] = g[p].diff(3).fillna(0)
        base = g[p].transform(lambda s: s.shift(1).rolling(24, min_periods=4).median())
        c[f"{p}_dev"] = (c[p] - base).fillna(0)
        c[f"{p}_sd6"] = g[p].transform(lambda s: s.rolling(6, min_periods=2).std()).fillna(0)
    c["ecoli_3h"] = g["ecoli"].transform(lambda s: s.rolling(3, min_periods=1).sum())
    c["storm"], c["night"] = (c["cond"] == "storm").astype(int), (c["cond"] == "night").astype(int)
    flags = [f"{p}_flag" for p in PARAMS if f"{p}_flag" in c]
    c["n_flag"] = sum(c[f].isin(["stuck", "spike", "long_gap", "range"]).astype(int) for f in flags) if flags else 0
    for k, name in [("truck", "water_truck"), ("purif", "purification_unit"), ("well", "well"), ("river", "river_intake")]:
        c[k] = (c["source_type"] == name).astype(int)
    return c

SCALE = {"ph": 0.35, "turbidity": 0.6, "chlorine": 0.2, "metal": 3.0}     # 항목별 '의미 있는 편차' 크기(가정)
W = {"corrob": 40, "persist": 25, "health": 20, "nondrift": 15}           # 신뢰도 가중치 매트릭스(합 100)
CUT = {"High": 60, "Med": 35}                                            # High만 급수 중단 검토 요청으로 올린다


def prepare(df):
    c = add_features(clean(df))
    c["rule"] = rule_grade(c)
    return c


def confidence(c, alert):
    """경보(alert=True) 행마다 신뢰도 점수와 구성 요소를 계산한다."""
    dev = pd.DataFrame({p: (c[f"{p}_dev"].abs() > SCALE[p]) | (c[f"{p}_d3"].abs() > SCALE[p]) for p in SCALE})
    n_dev = dev.sum(axis=1) + (c["ecoli_3h"] >= 2)
    corrob = np.minimum(n_dev, 3) / 3
    a = pd.Series(alert.astype(int), index=c.index)
    run = a.groupby([c["source_id"], (a != a.groupby(c["source_id"]).shift()).cumsum()]).cumsum()
    persist = np.minimum(run, 3) / 3
    flagged = c["n_flag"].groupby(c["source_id"]).transform(lambda s: s.rolling(6, min_periods=1).max()) > 0
    health = 1 - flagged.astype(float)
    # 드리프트 의심: 한 항목만 24시간 기준선에서 벗어나고, 6시간 변동성이 작으며(매끄러운 변화) 대장균이 없다
    g = c.groupby("source_id")
    jitter = {p: g[f"{p}_d1"].transform(lambda s: s.rolling(6, min_periods=3).std()).fillna(0) for p in SCALE}
    smooth = pd.concat([jitter[p] < SCALE[p] * 0.5 for p in SCALE], axis=1).all(axis=1)   # 변화량이 일정 = 기울기만 있는 변화
    nondrift = 1 - ((n_dev <= 1) & smooth & (c["ecoli_3h"] < 2)).astype(float)
    score = W["corrob"] * corrob + W["persist"] * persist + W["health"] * health + W["nondrift"] * nondrift
    level = np.where(score >= CUT["High"], "High", np.where(score >= CUT["Med"], "Med", "Low"))
    return pd.DataFrame({"score": score.round(1), "level": level, "n_dev": n_dev}, index=c.index)


train = pd.concat([prepare(make_dataset(seed=s, n_events=14, n_faults=14)) for s in (7, 8)], ignore_index=True)
model = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced_subsample", random_state=0, n_jobs=1)
model.fit(train[FEATS], train["grade"])

test = prepare(make_dataset(seed=11, n_events=10, n_faults=14))
alert = ((test["rule"] == 2) | (model.predict(test[FEATS]) == 2)).to_numpy()   # Baseline 또는 모델이 위험 판정
conf = confidence(test, alert)
A = test[alert].join(conf)
A["real"] = A["grade"] == 2
far_before = (~A["real"]).mean()
stop = A[A["level"] == "High"]
far_after = (~stop["real"]).mean()
cut = 1 - far_after / far_before
print(f"[교육용가상자료] 위험 경보 {len(A)}건(실제 위험 {int(A['real'].sum())}건, 오경보 {int((~A['real']).sum())}건)")
print(f"급수 중단 검토 경보 오경보율(FAR): 신뢰도 적용 전 {far_before:.1%} -> High만 상신 {far_after:.1%} "
      f"({cut:.0%} 감축, 목표 40% 이상: {'달성' if cut >= 0.4 else '미달'})")
print("\n[신뢰도 등급별 경보 수와 조치]")
tab = A.groupby("level").agg(경보=("real", "size"), 실제위험=("real", "sum"), 평균점수=("score", "mean"))
tab["조치"] = tab.index.map({"High": "급수 중단 검토 요청(HITL)", "Med": "즉시 재채수·현장 키트 검사", "Low": "센서 점검·재측정"})
print(tab.reindex(["High", "Med", "Low"]).round(1).to_string())
print("오경보의 원인(센서 고장 유형) x 등급:")
print(pd.crosstab(A.loc[~A["real"], "fault"], A.loc[~A["real"], "level"]).to_string())
miss_real = int((A["real"] & (A["level"] == "Low")).sum())
print(f"실제 위험인데 Low로 낮춰진 경보 {miss_real}건 -> Low도 재측정 대상이므로 무시되지 않는다")

auc = roc_auc_score(A["real"], A["score"])
print(f"\nROC-AUC(신뢰도 점수로 실제 위험 구분) {auc:.3f}")
print("[임계값 민감도] High 기준 / 상신 경보 / 오경보율 / 실제 위험 상신 비율")
for th in [45, 55, 60, 70, 80]:
    s = A[A["score"] >= th]
    print(f"  {th:>3} / {len(s):>4} / {(~s['real']).mean():6.1%} / {s['real'].sum() / A['real'].sum():6.1%}")

# 산출 일관성: 같은 사건·고장 조건에서 측정 잡음만 바꿔 5회 재산출했을 때 실제 위험 경보의 등급 일치율
EV = [(6, "sewage", 30, 14, 1.0), (4, "metal", 60, 12, 1.0), (2, "chemical", 40, 10, 1.0), (5, "creeping_metal", 20, 24, 1.0)]
FT = [(1, "drift", "turbidity", 24, 18), (0, "stuck", "chlorine", 30, 10)]
levels = []
for s in range(5):
    c = prepare(make_dataset(days=4, seed=100 + s, events=EV, faults=FT))
    al = ((c["rule"] == 2) | (model.predict(c[FEATS]) == 2)).to_numpy()
    levels.append(confidence(c, al)["level"].where(al, "-").to_numpy())
L = np.array(levels)
ref = pd.Series(L[0]).mode().iloc[0]
mask = (L != "-").sum(axis=0) >= 3
agree = np.mean([pd.Series(col[col != "-"]).value_counts(normalize=True).iloc[0] for col in L[:, mask].T])
print(f"\n잡음 재현 5회 등급 일치율 {agree:.1%} (경보 3회 이상 시점 {int(mask.sum())}개, 목표 90% 이상: {'달성' if agree >= 0.9 else '미달'})")
print("※ 신뢰도 등급은 판단 보조다. 급수 중단·재개는 의무(예방의학)·환경 담당 확인과 지휘관 승인으로 결정한다.")

fpr, tpr, _ = roc_curve(A["real"], A["score"])
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr, color="tab:blue"); ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[0].set_title(f"ROC of confidence score (AUC {auc:.2f})"); ax[0].set_xlabel("false positive rate"); ax[0].set_ylabel("true positive rate")
ax[1].hist([A.loc[A["real"], "score"], A.loc[~A["real"], "score"]], bins=12, label=["real danger", "false alarm"], color=["tab:red", "tab:gray"])
ax[1].axvline(CUT["High"], color="k", ls="--"); ax[1].set_xlabel("confidence score"); ax[1].legend(); ax[1].set_title("Score distribution (synthetic)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 위험 경보 262건(실제 위험 240건, 오경보 22건)
급수 중단 검토 경보 오경보율(FAR): 신뢰도 적용 전 8.4% -> High만 상신 0.7% (91% 감축, 목표 40% 이상: 달성)

[신뢰도 등급별 경보 수와 조치]
        경보  실제위험  평균점수                 조치
level                                    
High   137   136  84.1  급수 중단 검토 요청(HITL)
Med    112   100  53.5    즉시 재채수·현장 키트 검사
Low     13     4  27.5          센서 점검·재측정
오경보의 원인(센서 고장 유형) x 등급:
level  High  Low  Med
fault                
drift     1    0   10
none      0    9    2
실제 위험인데 Low로 낮춰진 경보 4건 -> Low도 재측정 대상이므로 무시되지 않는다

ROC-AUC(신뢰도 점수로 실제 위험 구분) 0.821
[임계값 민감도] High 기준 / 상신 경보 / 오경보율 / 실제 위험 상신 비율
   45 /  243 /   4.9% /  96.2%
   55 /  210 /   5.2% /  82.9%
   60 /  137 /   0.7% /  56.7%
   70 /  134 /   0.7% /  55.4%
   80 /   88 /   1.1% /  36.2%

잡음 재현 5회 등급 일치율 93.6% (경보 3회 이상 시점 99개, 목표 90% 이상: 달성)
※ 신뢰도 등급은 판단 보조다. 급수 중단·재개는 의무(예방의학)·환경 담당 확인과 지휘관 승인으로 결정한다.
```
</details>
